# Build Cosmograph CSV Files from Adjacency Matrix
## From : `inter_to_ict_chb01_03_2980_3010_adjacency_sparse.npz`

This notebook builds **6 Cosmograph-compatible CSV file pairs** directly from the sparse adjacency matrix :

| File pair | Nodes | Edges | Purpose |
|-----------|-------|-------|---------|
| cosmo_pre_nodes/edges.csv | 5,888 | 302 | Static pre-ictal (intra + inter, thresholded) |
| cosmo_ict_nodes/edges.csv | 5,888 | 4,342 | Static ictal (intra + inter, thresholded) |
| cosmograph_nodes/edges_CZ.csv | 7,680 | 1,458 | CZ electrode streaming (timeline) |
| cosmograph_nodes/edges_all.csv | 176,640 | 33,419 | All 23 electrodes streaming (timeline) |


## Step 1 — Install & Import

In [ ]:
import numpy as np
import math
import csv
import os
from scipy import sparse
from scipy.sparse import csr_matrix

## Step 2 — Constants & Labels

In [ ]:
N_ELEC   = 23     # number of EEG electrodes
N_TIME   = 7680   # total timepoints (30s × 256 Hz)
WINDOW   = 256    # timepoints per 1-second window
N_WIN    = N_TIME // WINDOW   # 30 windows
SEIZURE  = 15     # seizure onset window index (t = 15s)

# Electrode labels (10-20 system)
LABELS = ['FP1', 'FP2', 'F7', 'F3', 'FZ', 'F4', 'F8', 'T7', 'C3', 'CZ', 'C4', 'T8', 'P7', 'P3', 'PZ', 'P4', 'P8', 'O1', 'OZ', 'O2', 'A1', 'A2', 'T9']

# Electrode positions in head layout (normalized 0-1, canvas 5000×5000)
ELEC_POS = [[0.5, 0.05], [0.55, 0.05], [0.15, 0.18], [0.35, 0.15], [0.5, 0.12], [0.65, 0.15], [0.85, 0.18], [0.1, 0.45], [0.3, 0.38], [0.5, 0.35], [0.7, 0.38], [0.9, 0.45], [0.1, 0.7], [0.3, 0.62], [0.5, 0.6], [0.7, 0.62], [0.9, 0.7], [0.3, 0.85], [0.5, 0.82], [0.7, 0.85], [0.05, 0.55], [0.95, 0.55], [0.05, 0.75]]

CANVAS   = 5000   # canvas size for head layout
R_ELEC   = 80     # radius of each electrode's circular VG
R_CZ     = 500    # radius for single-electrode (CZ) circular layout

# ── Thresholds ──
INTRA_THR = 0.2   # intra-electrode: edge must appear in ≥ 3/15 windows
INTER_THR = 0.5   # inter-electrode: Pearson correlation of degree sequences ≥ 0.5

print(f" Constants set")
print(f" {N_ELEC} electrodes × {N_WIN} windows × {WINDOW} timepoints = {N_ELEC*N_WIN*WINDOW:,} nodes")
print(f" Seizure onset: window {SEIZURE} (t = 15s)")
print(f" Intra threshold: frequency ≥ {INTRA_THR}")
print(f" Inter threshold: correlation ≥ {INTER_THR}")

## Step 3 — Load & Parse Adjacency Matrix

In [ ]:
NPZ_PATH = 'inter_to_ict_chb01_03_2980_3010_adjacency_sparse.npz'
mat = sparse.load_npz(NPZ_PATH)
cx  = mat.tocoo()

print(f"Matrix shape    : {{mat.shape}}")
print(f"Non-zero entries: {{mat.nnz:,}}")

# Node k → electrode = k // N_TIME, timepoint = k % N_TIME
rows_elec = cx.row // N_TIME   # electrode index of source node
rows_time = cx.row % N_TIME    # timepoint of source node
cols_elec = cx.col // N_TIME   # electrode index of target node

same  = rows_elec == cols_elec   # True = intra-electrode (actual HVG edges)
cross = ~same                     # True = cross-electrode (always fully connected)

print(f"\nIntra-electrode edges (HVG): {{same.sum():,}}")
print(f"Cross-electrode edges       : {{cross.sum():,}}")

## Step 4 — Compute VG Degree for Every Node

In [ ]:
intra_mat = csr_matrix(
    (np.ones(same.sum()), (cx.row[same], cx.col[same])),
    shape=mat.shape
)

# Degree = row sum + column sum (undirected graph)
intra_deg = (
    np.array(intra_mat.sum(axis=1)).flatten() +
    np.array(intra_mat.sum(axis=0)).flatten()
)

# Reshape to (electrode, timepoint)
deg_full = intra_deg.reshape(N_ELEC, N_TIME)  # shape: (23, 7680)

print(f"Degree matrix shape: {{deg_full.shape}}")
print(f"\nMean degree:")
print(f"  Pre-ictal (w=0-14) : {{deg_full[:, :SEIZURE*WINDOW].mean():.4f}}")
print(f"  Ictal     (w=15-29): {{deg_full[:, SEIZURE*WINDOW:].mean():.4f}}")
print(f"\nTop 5 electrodes by ictal mean degree:")
ict_means = deg_full[:, SEIZURE*WINDOW:].mean(axis=1)
for ei in np.argsort(ict_means)[::-1][:5]:
    pre = deg_full[ei, :SEIZURE*WINDOW].mean()
    ict = deg_full[ei, SEIZURE*WINDOW:].mean()
    inc = f"{{ict/pre:.1f}}x" if pre > 0 else "∞"
    print(f"  {{LABELS[ei]:<4}}: pre={{pre:.3f}}  ict={{ict:.3f}}  increase={{inc}}")

## Step 5 — Compute Inter-electrode Correlation

For static files: Pearson correlation between each electrode pair's
**mean VG degree sequence** (length-15 vector, one value per window).


In [ ]:
def compute_inter_corr(w_start, w_end):
    """Pearson correlation of mean VG degree per window between electrode pairs."""
    n_wins = w_end - w_start
    # Mean degree per window per electrode → shape (N_ELEC, n_wins)
    deg_wins = deg_full[:, w_start*WINDOW:w_end*WINDOW].reshape(N_ELEC, n_wins, WINDOW).mean(axis=2)

    corr = np.zeros((N_ELEC, N_ELEC))
    for i in range(N_ELEC):
        for j in range(i+1, N_ELEC):
            a, b = deg_wins[i], deg_wins[j]
            ma, mb = a.mean(), b.mean()
            na, nb = a - ma, b - mb
            denom = np.sqrt((na**2).sum() * (nb**2).sum())
            if denom == 0:
                continue
            r = max(0, float((na * nb).sum() / denom))
            corr[i][j] = corr[j][i] = round(r, 4)
    return corr

corr_pre = compute_inter_corr(0, SEIZURE)
corr_ict = compute_inter_corr(SEIZURE, N_WIN)

pre_pairs = (corr_pre >= INTER_THR).sum() // 2
ict_pairs = (corr_ict >= INTER_THR).sum() // 2

print(f"Inter-electrode correlation:")
print(f"  Pre-ictal mean : {{corr_pre[corr_pre>0].mean():.3f}}")
print(f"  Ictal mean     : {{corr_ict[corr_ict>0].mean():.3f}}")
print(f"  Pre pairs ≥ {{INTER_THR}}: {{pre_pairs}}")
print(f"  Ict pairs ≥ {{INTER_THR}}: {{ict_pairs}}")

## Step 6 — Build Static CSV Files (Pre-ictal & Ictal)

**Node layout:** Each electrode's 256 timepoints arranged in a small circle,
positioned at anatomical head location (10-20 system).

**Edges:**
- `intra`: VG frequency ≥ 0.2 (appeared in ≥ 3/15 windows)
- `inter`: Pearson correlation ≥ 0.5 (connecting midpoint nodes t=128)


In [ ]:
def build_static_csvs(phase):
    """Build node + edge CSVs for one phase (pre-ictal or ictal)."""
    is_pre  = (phase == 'pre')
    w_start = 0       if is_pre else SEIZURE
    w_end   = SEIZURE if is_pre else N_WIN
    n_wins  = w_end - w_start
    corr    = corr_pre if is_pre else corr_ict

    nodes = []
    edges = []

    for ei in range(N_ELEC):
        ex = ELEC_POS[ei][0] * CANVAS
        ey = ELEC_POS[ei][1] * CANVAS

        # Mean degree over 15 windows for this electrode
        if is_pre:
            avg_deg = deg_full[ei, :SEIZURE*WINDOW].reshape(n_wins, WINDOW).mean(axis=0)
        else:
            avg_deg = deg_full[ei, SEIZURE*WINDOW:].reshape(n_wins, WINDOW).mean(axis=0)

        # ── Nodes ──
        for k in range(WINDOW):
            angle = (k / WINDOW) * 2 * math.pi - math.pi / 2
            x = round(ex + R_ELEC * math.cos(angle), 2)
            y = round(ey + R_ELEC * math.sin(angle), 2)
            nodes.append({
                'id':           f"e{{ei}}_t{{k}}",
                'x':            x,
                'y':            y,
                'degree':       round(float(avg_deg[k]), 4),
                'electrode':    LABELS[ei],
                'electrode_id': ei,
                'phase':        phase,
                'timepoint':    k
            })

        # ── Intra edges (frequency threshold) ──
        edge_count = {{}}
        for w in range(w_start, w_end):
            t_start = w * WINDOW
            t_end   = (w + 1) * WINDOW
            mask = (same & (rows_elec == ei) &
                    (rows_time >= t_start) & (rows_time < t_end))
            r_tp = (cx.row[mask] % N_TIME) - t_start
            c_tp = (cx.col[mask] % N_TIME) - t_start
            seen = set()
            for ri, ci in zip(r_tp.tolist(), c_tp.tolist()):
                if ri >= ci or ri < 0 or ci >= WINDOW:
                    continue
                if (ri, ci) in seen:
                    continue
                seen.add((ri, ci))
                edge_count[(ri, ci)] = edge_count.get((ri, ci), 0) + 1

        for (a, b), cnt in edge_count.items():
            freq = cnt / n_wins
            if freq >= INTRA_THR:
                edges.append({{
                    'source':    f"e{{ei}}_t{{a}}",
                    'target':    f"e{{ei}}_t{{b}}",
                    'weight':    round(freq, 4),
                    'type':      'intra',
                    'electrode': LABELS[ei]
                }})

    # ── Inter edges (correlation threshold) ──
    for i in range(N_ELEC):
        for j in range(i + 1, N_ELEC):
            c = corr[i][j]
            if c >= INTER_THR:
                edges.append({{
                    'source':    f"e{{i}}_t128",
                    'target':    f"e{{j}}_t128",
                    'weight':    round(c, 4),
                    'type':      'inter',
                    'electrode': f"{{LABELS[i]}}-{{LABELS[j]}}"
                }})

    return nodes, edges

nodes_pre, edges_pre = build_static_csvs('pre')
nodes_ict, edges_ict = build_static_csvs('ict')

intra_pre = sum(1 for e in edges_pre if e['type'] == 'intra')
inter_pre = sum(1 for e in edges_pre if e['type'] == 'inter')
intra_ict = sum(1 for e in edges_ict if e['type'] == 'intra')
inter_ict = sum(1 for e in edges_ict if e['type'] == 'inter')

print(f"Pre-ictal: {{len(nodes_pre):,}} nodes | {{len(edges_pre):,}} edges (intra={{intra_pre}}, inter={{inter_pre}})")
print(f"Ictal    : {{len(nodes_ict):,}} nodes | {{len(edges_ict):,}} edges (intra={{intra_ict}}, inter={{inter_ict}})")
print(f"Edge increase: {{len(edges_ict)/len(edges_pre):.1f}}x")

## Step 7 — Build CZ Streaming CSV (Timeline)

In [ ]:
def build_cz_csvs(ei_cz=9):
    """CZ electrode: 30 windows × 256 timepoints, timeline-enabled."""
    nodes = []
    edges = []

    for w in range(N_WIN):
        t_start = w * WINDOW
        t_end   = (w + 1) * WINDOW
        phase   = 'pre-ictal' if w < SEIZURE else 'ictal'

        for k in range(WINDOW):
            tp    = t_start + k
            d     = float(deg_full[ei_cz, tp])
            angle = (k / WINDOW) * 2 * math.pi - math.pi / 2
            x     = round(R_CZ * math.cos(angle), 2)
            y     = round(R_CZ * math.sin(angle), 2)
            nodes.append({{
                'id':        f"w{{w}}_t{{k}}",
                'x':         x,
                'y':         y,
                'degree':    round(d, 4),
                'time':      w,       # ← Cosmograph timeline
                'phase':     phase,
                'timepoint': k
            }})

        # VG edges for this window
        mask  = (same & (rows_elec == ei_cz) &
                 (rows_time >= t_start) & (rows_time < t_end))
        r_tp  = (cx.row[mask] % N_TIME) - t_start
        c_tp  = (cx.col[mask] % N_TIME) - t_start
        seen  = set()
        for ri, ci in zip(r_tp.tolist(), c_tp.tolist()):
            if ri >= ci or ri < 0 or ci >= WINDOW:
                continue
            if (ri, ci) in seen:
                continue
            seen.add((ri, ci))
            dist = ci - ri
            edges.append({{
                'source': f"w{{w}}_t{{ri}}",
                'target': f"w{{w}}_t{{ci}}",
                'weight': round(1 / dist, 4),
                'time':   w            # ← Cosmograph timeline
            }})

    return nodes, edges

nodes_cz, edges_cz = build_cz_csvs(ei_cz=9)  # CZ = index 9
pre_e_cz = sum(1 for e in edges_cz if e['time'] < SEIZURE)
ict_e_cz = sum(1 for e in edges_cz if e['time'] >= SEIZURE)

print(f"CZ streaming: {{len(nodes_cz):,}} nodes | {{len(edges_cz):,}} total edges")
print(f"  Pre-ictal edges: {{pre_e_cz}} (windows 0-14)")
print(f"  Ictal edges    : {{ict_e_cz}} (windows 15-29)")
print(f"  Edge increase  : {{ict_e_cz/pre_e_cz:.1f}}x" if pre_e_cz > 0 else "  Edge increase: ∞")

## Step 8 — Build All-Electrodes Streaming CSV (Timeline)

In [ ]:
def build_all_csvs():
    """All 23 electrodes: 30 windows × 256 timepoints, head layout, timeline-enabled."""
    nodes = []
    edges = []

    for w in range(N_WIN):
        t_start = w * WINDOW
        t_end   = (w + 1) * WINDOW
        phase   = 'pre-ictal' if w < SEIZURE else 'ictal'

        for ei in range(N_ELEC):
            ex = ELEC_POS[ei][0] * CANVAS
            ey = ELEC_POS[ei][1] * CANVAS

            for k in range(WINDOW):
                tp    = t_start + k
                d     = float(deg_full[ei, tp])
                angle = (k / WINDOW) * 2 * math.pi - math.pi / 2
                x     = round(ex + R_ELEC * math.cos(angle), 2)
                y     = round(ey + R_ELEC * math.sin(angle), 2)
                nodes.append({{
                    'id':        f"e{{ei}}_w{{w}}_t{{k}}",
                    'x':         x,
                    'y':         y,
                    'degree':    round(d, 4),
                    'time':      w,         # ← Cosmograph timeline
                    'phase':     phase,
                    'electrode': LABELS[ei],
                    'timepoint': k
                }})

            # VG edges for this electrode in this window
            mask  = (same & (rows_elec == ei) &
                     (rows_time >= t_start) & (rows_time < t_end))
            r_tp  = (cx.row[mask] % N_TIME) - t_start
            c_tp  = (cx.col[mask] % N_TIME) - t_start
            seen  = set()
            for ri, ci in zip(r_tp.tolist(), c_tp.tolist()):
                if ri >= ci or ri < 0 or ci >= WINDOW:
                    continue
                if (ri, ci) in seen:
                    continue
                seen.add((ri, ci))
                dist = ci - ri
                edges.append({{
                    'source':    f"e{{ei}}_w{{w}}_t{{ri}}",
                    'target':    f"e{{ei}}_w{{w}}_t{{ci}}",
                    'weight':    round(1 / dist, 4),
                    'time':      w,         # ← Cosmograph timeline
                    'electrode': LABELS[ei]
                }})

        if w % 5 == 0:
            print(f"  Window {{w:2d}} done — nodes so far: {{len(nodes):,}}")

    return nodes, edges

print("Building all-electrodes streaming CSV...")
nodes_all, edges_all = build_all_csvs()

print(f"\nAll-electrodes streaming: {{len(nodes_all):,}} nodes | {{len(edges_all):,}} edges")

## Step 9 — Save All CSV Files

In [ ]:
def write_csv(path, rows, fields):
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)
    kb = os.path.getsize(path) // 1024
    print(f"  ✓ {{os.path.basename(path)}}: {{len(rows):,}} rows | {{kb}} KB")

print("Saving static files...")
write_csv('cosmo_pre_nodes.csv', nodes_pre,
          ['id','x','y','degree','electrode','electrode_id','phase','timepoint'])
write_csv('cosmo_pre_edges.csv', edges_pre,
          ['source','target','weight','type','electrode'])
write_csv('cosmo_ict_nodes.csv', nodes_ict,
          ['id','x','y','degree','electrode','electrode_id','phase','timepoint'])
write_csv('cosmo_ict_edges.csv', edges_ict,
          ['source','target','weight','type','electrode'])

print("\nSaving CZ streaming files...")
write_csv('cosmograph_nodes_CZ.csv', nodes_cz,
          ['id','x','y','degree','time','phase','timepoint'])
write_csv('cosmograph_edges_CZ.csv', edges_cz,
          ['source','target','weight','time'])

print("\nSaving all-electrodes streaming files...")
write_csv('cosmograph_nodes_all.csv', nodes_all,
          ['id','x','y','degree','time','phase','electrode','timepoint'])
write_csv('cosmograph_edges_all.csv', edges_all,
          ['source','target','weight','time','electrode'])